# PySpark window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-window-functions.ipynb)

"What is each customer's largest order?", "how many days since their previous order?", "what is the running
total?", "keep only the latest version of each record". A `groupBy` cannot answer these, because it collapses the
rows you need to keep. A **window function** keeps every row and adds a value computed from related rows.

This page covers every window function in `pyspark.sql.functions` and the patterns built from them: aggregates
over a partition, ranking, top-N, deduplication, `lag` and `lead`, `first`, `last` and `nth_value`, frames,
running and moving totals, forward fill, sessions, gaps-and-islands, the SQL syntax, and how Spark executes a window.

Each section explains how something works, where it matters and what to watch out for, then runs small examples.
Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb). The window
sections of [PySpark 7](pyspark-07-aggregations-and-windows.ipynb) cover the same ideas more briefly.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has
no region) and `orders` (one row per order), the slice of the orders platform every module shares, introduced in
[PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data). Later sections add small tables of their own
where a pattern needs a particular shape of data.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## How a window works

A window function needs two things: the **function** (what to compute) and the **window specification** (over
which rows). You build the specification with `Window` and attach it with `.over(spec)`:

```python
spec = (
    Window.partitionBy("customer_id")  # which rows belong together
    .orderBy("order_ts")               # their order inside each partition
    .rowsBetween(-2, 0)                # optional frame: which neighbours each row sees
)
F.avg("amount").over(spec)
```

| part | question it answers | if you leave it out |
| --- | --- | --- |
| `partitionBy(cols)` | which rows are compared with each other? | the whole DataFrame is one partition |
| `orderBy(cols)` | in what order? | no order: ranking, `lag` and `lead` are not allowed |
| `rowsBetween` / `rangeBetween` | which rows around the current one? | a default frame (see [Frames](#frames-choose-which-rows-each-row-sees)) |

Three families of functions can go before `.over(...)`:

| family | functions | needs `orderBy` |
| --- | --- | --- |
| ranking | `row_number`, `rank`, `dense_rank`, `percent_rank`, `cume_dist`, `ntile` | yes |
| offset (analytic) | `lag`, `lead`, `nth_value`, `first`, `last`, `first_value`, `last_value` | yes for `lag`, `lead`, `nth_value`; recommended for the rest |
| aggregate | any aggregate: `sum`, `avg`, `count`, `min`, `max`, `collect_list`, `max_by`, ... | no |

**Where it matters**
- **Per-row context:** each order next to its customer's total, rank or previous order.
- **Replacing self-joins:** "compare with the previous row" used to need a join on `row_number - 1`.

**Watch out for**
- `groupBy` returns one row per group; a window returns every input row. Choose by the shape of the output you need.
- You cannot filter on a window function in the same `where`: compute it in a column first, then filter.

### Example

Compute each customer's revenue with `groupBy`, then the same total with a window, and compare the shapes.

In [ ]:
orders.groupBy("customer_id").agg(F.sum("amount").alias("customer_total")).orderBy("customer_id").show(3)

by_customer = Window.partitionBy("customer_id")
orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(by_customer).alias("customer_total"),
).orderBy("customer_id", "order_id").show(5)

**What to notice:** `groupBy` returned one row per customer and lost the order ids. The window returned all twelve
orders and repeated the customer's total (234.50 for customer 1) on each of its three rows, so you can compare
every order with its total without a join.

## Aggregates over a whole partition

Any aggregate function becomes a window function with `.over(spec)`. With only `partitionBy`, every row sees its
whole partition, so the result is the same on each row of a partition: the group total, average, maximum or count.
Typical uses are **shares** (row / total), **differences from the group** (row - average) and **filters against
the group** (rows above their group's average).

**Where it matters**
- **Contribution analysis:** each product's share of its category's revenue.
- **Outlier checks:** orders more than three times their customer's average.

**Watch out for**
- Add `orderBy` and the aggregate silently becomes a running aggregate, because the default frame changes. Keep
  group-level aggregates on a specification with no order.

### Example

Put the customer's total, average, largest order and order count on each order, then keep only orders above their
customer's average.

In [ ]:
with_context = orders.select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(by_customer).alias("total"),
    F.round(F.avg("amount").over(by_customer), 2).alias("avg"),
    F.max("amount").over(by_customer).alias("largest"),
    F.count("*").over(by_customer).alias("orders"),
    F.round(F.col("amount") / F.sum("amount").over(by_customer) * 100, 1).alias("pct_of_customer"),
    (F.col("amount") - F.avg("amount").over(by_customer)).cast("decimal(12,2)").alias("vs_avg"),
)
with_context.orderBy("customer_id", "order_id").show(6)
with_context.where("amount > avg").select("customer_id", "order_id", "amount", "avg").orderBy("customer_id").show()

**What to notice:** Customer 1's orders are 51.2, 36.0 and 12.8 percent of its 234.50 total, and `vs_avg` shows how
far each one is from the 78.17 average. The second table filters on the window result, which only works because
the window was computed into a column first. Customer 5 has a single order, which equals its own average, so it
does not appear.

## Rank rows: row_number, rank and dense_rank

All three number rows within each partition in the window's order. They differ only on **ties**:

| function | values for 300, 300, 200, 100 | use when |
| --- | --- | --- |
| `row_number()` | 1, 2, 3, 4 | you need exactly one row per position (dedup, top-1) |
| `rank()` | 1, 1, 3, 4 | ties share a place and the next place is skipped, as in sports |
| `dense_rank()` | 1, 1, 2, 3 | ties share a place and no number is skipped ("top 2 values") |

**Where it matters**
- **Leaderboards** per region, store or category.
- **Exactly one row per key** with `row_number`.

**Watch out for**
- With ties, `row_number` picks an order you cannot rely on: two runs can number the tied rows differently. Add
  a unique column to the `orderBy` as a tie-breaker.
- Nulls sort first in ascending order; use `F.desc_nulls_last` or `F.asc_nulls_last` when they should rank last.

### Example

Two north reps tie at 300. Compare the three functions, with and without a tie-breaker.

In [ ]:
sales = spark.createDataFrame(
    [("north", "Ana", 300), ("north", "Ben", 300), ("north", "Cai", 200), ("north", "Dee", 100),
     ("south", "Eli", 500), ("south", "Fay", None)],
    "region string, rep string, revenue int",
)
by_revenue = Window.partitionBy("region").orderBy(F.desc_nulls_last("revenue"))
by_revenue_stable = Window.partitionBy("region").orderBy(F.desc_nulls_last("revenue"), "rep")
sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue).alias("row_number"),
    F.row_number().over(by_revenue_stable).alias("row_number_stable"),
    F.rank().over(by_revenue).alias("rank"),
    F.dense_rank().over(by_revenue).alias("dense_rank"),
).orderBy("region", "row_number_stable").show()

**What to notice:** Ana and Ben share `rank` and `dense_rank` 1. After them `rank` jumps to 3 and `dense_rank`
continues with 2. `row_number` must separate them; only `row_number_stable`, ordered by `rep` as well, guarantees
Ana is 1 on every run. Fay has no revenue, and `desc_nulls_last` ranks her last in the south rather than first.

## Relative position: percent_rank, cume_dist and ntile

| function | formula | values for 4 rows |
| --- | --- | --- |
| `percent_rank()` | (rank - 1) / (rows - 1) | 0, 0.33, 0.67, 1 |
| `cume_dist()` | rows at or before this one / rows | 0.25, 0.5, 0.75, 1 |
| `ntile(n)` | splits the ordered rows into `n` buckets as evenly as possible | `ntile(2)`: 1, 1, 2, 2 |

**Where it matters**
- **Segmentation:** quartiles of customers by spend (`ntile(4)`), the top 10 percent (`percent_rank`).
- **Percentiles per group** without collecting the data.

**Watch out for**
- When rows do not divide evenly, `ntile` gives the extra rows to the first buckets.
- Ties share a `percent_rank` and a `cume_dist`, but `ntile` may put tied rows in different buckets.

### Example

Split all orders into spend quartiles and show each order's relative position.

In [ ]:
by_amount_all = Window.orderBy("amount")  # a single partition: fine for 12 rows, see the performance section
orders.select(
    "order_id", "amount",
    F.ntile(4).over(by_amount_all).alias("quartile"),
    F.round(F.percent_rank().over(by_amount_all), 2).alias("percent_rank"),
    F.round(F.cume_dist().over(by_amount_all), 2).alias("cume_dist"),
).orderBy("amount").show()

**What to notice:** Twelve orders split into four quartiles of three. The smallest order has `percent_rank` 0 and
the largest 1, while `cume_dist` starts at 1/12 = 0.08 and also ends at 1. Order 9 (1,180.00) is alone in the top
of the distribution, so a "top 10 percent" filter on `percent_rank >= 0.9` keeps only it and order 7.

## Keep the top N rows per group

Rank inside each partition, then filter on the rank. The ranking function decides what "top 2" means when there
are ties: `row_number` returns exactly two rows, `rank` and `dense_rank` return every row tied for a place.

**Where it matters**
- **Best sellers per category,** largest orders per customer, highest-paid employee per department.

**Watch out for**
- Filter in a separate step after the window column exists.
- Decide with the business whether ties should all be kept; it changes how many rows a report has.

### Example

Keep each region's top two reps with each of the three ranking functions.

In [ ]:
ranked = sales.select(
    "region", "rep", "revenue",
    F.row_number().over(by_revenue_stable).alias("rn"),
    F.rank().over(by_revenue).alias("rnk"),
    F.dense_rank().over(by_revenue).alias("drnk"),
)
print("row_number <= 2:", [r.rep for r in ranked.where("rn <= 2").orderBy("region", "rn").collect()])
print("rank <= 2:      ", [r.rep for r in ranked.where("rnk <= 2").orderBy("region", "rnk", "rep").collect()])
print("dense_rank <= 2:", [r.rep for r in ranked.where("drnk <= 2").orderBy("region", "drnk", "rep").collect()])

**What to notice:** `row_number` gives exactly two reps per region. `rank <= 2` gives the same north reps because
Ana and Ben fill places 1 and 2, but `dense_rank <= 2` also brings in Cai, the second-highest *value*. In the south
all three agree, with Eli and Fay.

## Deduplicate: keep the latest record per key

Change feeds, retries and replays deliver several versions of one key. Number each key's versions with
`row_number` ordered newest first, by the **source** sequence (log sequence number, version, the source's
`updated_at`) plus a stable tie-breaker, and keep `rn = 1`.

`dropDuplicates(["key"])` also keeps one row per key, but which row it keeps is not defined.

**Where it matters**
- **Silver tables** built from CDC: one current row per order.
- **Reprocessing:** a replayed batch must pick the same winners as the first run.

**Watch out for**
- Ingestion time changes on a replay; use it only as the last tie-breaker.
- `max_by(status, lsn)` in a `groupBy` gives the latest value of one column; the window keeps the whole row.

### Example

Order 101 has three versions and order 102's final change arrived twice. Keep the latest version of each.

In [ ]:
changes = spark.createDataFrame(
    [(101, "placed", 1, datetime(2026, 1, 5, 9, 0)), (101, "paid", 2, datetime(2026, 1, 5, 9, 5)),
     (101, "shipped", 3, datetime(2026, 1, 6, 8, 0)), (102, "placed", 1, datetime(2026, 1, 5, 10, 0)),
     (102, "cancelled", 2, datetime(2026, 1, 5, 10, 1)), (102, "cancelled", 2, datetime(2026, 1, 5, 10, 9))],
    "order_id int, status string, source_lsn long, ingested_at timestamp",
)
newest_first = Window.partitionBy("order_id").orderBy(F.desc("source_lsn"), F.desc("ingested_at"))
latest = changes.withColumn("rn", F.row_number().over(newest_first)).where("rn = 1").drop("rn")
latest.orderBy("order_id").show()
changes.groupBy("order_id").agg(F.max_by("status", "source_lsn").alias("latest_status")).orderBy("order_id").show()

**What to notice:** Order 101 keeps `shipped`, its highest sequence. Order 102's two `cancelled` rows share sequence
2, so ingestion time breaks the tie and the 10:09 copy wins on every run. `max_by` reaches the same statuses in a
`groupBy`, but returns only that column, not the whole winning row.

## Look back and ahead: lag and lead

`F.lag(col, offset=1, default=None)` reads the value `offset` rows **before** the current row in window order;
`F.lead(col, offset=1, default=None)` reads it `offset` rows **after**. At the edge of a partition there is no such
row, so they return `default` (null unless you pass one).

**Where it matters**
- **Gaps:** days since the previous order, minutes between events.
- **Change:** difference or percentage change from the previous value; the next status after this one.

**Watch out for**
- Without `partitionBy`, `lag` reads the previous *customer's* row at each boundary.
- Ties in the `orderBy` make "previous" ambiguous; add a tie-breaker.
- A `default` of 0 can hide that there was no previous row. Use it only when that is truly what 0 means.

### Example

For each order, read the previous order's time and amount, compute the gap and the change, and look two orders ahead.

In [ ]:
in_order = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")
orders.where("customer_id IN (1, 2, 3)").select(
    "customer_id", "order_id", "order_ts", "amount",
    F.lag("amount").over(in_order).alias("prev_amount"),
    (F.col("amount") - F.lag("amount").over(in_order)).alias("change"),
    F.round((F.unix_timestamp("order_ts") - F.unix_timestamp(F.lag("order_ts").over(in_order))) / 3600, 1)
     .alias("hours_since_prev"),
    F.lead("order_id").over(in_order).alias("next_order"),
    F.lead("order_id", 2, -1).over(in_order).alias("order_after_next"),
).orderBy("customer_id", "order_ts").show()

**What to notice:** Each customer's first order has no previous amount, so `change` and `hours_since_prev` are null
too; customer 1's orders were 0.8 and 1.6 hours apart. `lead(order_id, 2, -1)` looks two rows ahead and returns the
default -1 where that row does not exist, which is every row except customer 1's first.

### Example: detect changes and number the runs

A common use of `lag` is a **change flag**: 1 when a value differs from the previous row, else 0. A running sum
of the flags then gives each unbroken run of the same value its own number, so you can group runs.

In [ ]:
status_log = spark.createDataFrame(
    [("M1", 1, "up"), ("M1", 2, "up"), ("M1", 3, "down"), ("M1", 4, "down"), ("M1", 5, "up"),
     ("M2", 1, "down"), ("M2", 2, "up")],
    "machine string, minute int, state string",
)
by_minute = Window.partitionBy("machine").orderBy("minute")
runs = (
    status_log
    .withColumn("changed", F.when(F.lag("state").over(by_minute).eqNullSafe(F.col("state")), 0).otherwise(1))
    .withColumn("run_id", F.sum("changed").over(by_minute.rowsBetween(Window.unboundedPreceding, 0)))
)
runs.orderBy("machine", "minute").show()
runs.groupBy("machine", "run_id", "state").agg(
    F.min("minute").alias("from_minute"), F.max("minute").alias("to_minute")
).orderBy("machine", "run_id").show()

**What to notice:** The first row of each machine counts as a change because there is no previous state, so every
run, including the first, gets a number. Machine M1 has three runs: up for minutes 1 to 2, down for 3 to 4 and up
again at 5. Grouping by `run_id` turns row-level states into start and end times, the shape an uptime report needs.

## first, last and nth_value

| function | returns |
| --- | --- |
| `first(col, ignorenulls=False)`, `first_value` | the first value in the frame |
| `last(col, ignorenulls=False)`, `last_value` | the last value in the frame |
| `nth_value(col, n, ignoreNulls=False)` | the value in the `n`th row of the frame (1-based), or null if the frame is shorter |

All three read the **frame**, not the whole partition. With an `orderBy` and no explicit frame, the frame ends at
the current row, so `last` returns the current row's value and `nth_value(col, 2)` is null on the first row. Use
`rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` to see the whole partition.

**Where it matters**
- **First and latest values:** first order amount, current status, the second purchase (a classic retention metric).

**Watch out for**
- `ignorenulls=True` skips nulls, which is what forward fill uses; without it a null in the frame is returned as is.

### Example

Read each customer's first, last and second order with the default frame and with the whole-partition frame.

In [ ]:
whole = in_order.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
orders.where("customer_id IN (1, 2)").select(
    "customer_id", "order_id",
    F.first("order_id").over(in_order).alias("first_default"),
    F.last("order_id").over(in_order).alias("last_default"),
    F.last("order_id").over(whole).alias("last_whole"),
    F.nth_value("order_id", 2).over(in_order).alias("second_default"),
    F.nth_value("order_id", 2).over(whole).alias("second_whole"),
).orderBy("customer_id", "order_id").show()

**What to notice:** `first` is right with either frame because the frame always starts at the partition's first
row. `last_default` just repeats each row's own id, the classic trap; `last_whole` gives the real last order (3
and 5). `nth_value` with the default frame is null on the first row, which has not reached the second row yet;
with the whole frame every row sees the second order.

## Frames: choose which rows each row sees

A frame limits the rows of the partition that each row's function reads, relative to the current row.

| frame | meaning |
| --- | --- |
| `rowsBetween(start, end)` | physical rows: `-2, 0` is the two previous rows and this one |
| `rangeBetween(start, end)` | values of the single `orderBy` column: `-7, 0` is every row whose value is within 7 below |
| `Window.unboundedPreceding`, `Window.unboundedFollowing`, `Window.currentRow` | the partition start, the partition end, this row (0) |

The default frame depends on whether the window has an order:

| specification | default frame |
| --- | --- |
| `partitionBy` only | the whole partition |
| `partitionBy` + `orderBy` | `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` |

That `RANGE` default includes every row that **ties** with the current row in the ordering, which surprises people
writing running totals.

**Where it matters**
- **Running totals** (start to current row) and **moving windows** (a fixed number of neighbours).
- **Calendar windows** such as "the last 7 days", which need `rangeBetween` over a number.

**Watch out for**
- Ranking functions and `lag` / `lead` ignore frames.
- `rangeBetween` needs exactly one numeric (or date-as-number) `orderBy` column.

### Example

Two payments share a timestamp. Compare a running total with the default `RANGE` frame and with a `ROWS` frame.

In [ ]:
payments = spark.createDataFrame(
    [(1, "09:00", 10), (2, "09:05", 20), (3, "09:05", 30), (4, "09:10", 40)],
    "payment_id int, at string, amount int",
)
by_time = Window.orderBy("at")
payments.select(
    "payment_id", "at", "amount",
    F.sum("amount").over(by_time).alias("default_range"),
    F.sum("amount").over(by_time.rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias("rows_frame"),
    F.sum("amount").over(Window.orderBy("at", "payment_id").rowsBetween(Window.unboundedPreceding, 0))
     .alias("rows_with_tiebreak"),
).orderBy("payment_id").show()

**What to notice:** With the default `RANGE` frame, payments 2 and 3 both show 60, because rows that tie on `at`
enter the frame together. The `ROWS` frame adds one row at a time (30, then 60), but the order between the tied
rows is arbitrary unless you add a tie-breaker, as `rows_with_tiebreak` does.

## Running totals and cumulative metrics

A frame from `Window.unboundedPreceding` to `Window.currentRow` gives everything so far: running sum, running
count, running maximum, or a running average. Divide a running sum by the partition total for a cumulative share.

**Where it matters**
- **Cumulative revenue** per customer or per month, year-to-date figures.
- **Pareto analysis:** how many customers make up 80 percent of revenue.

**Watch out for**
- Order by something unique, or ties share a running value (the default `RANGE` frame) or are numbered arbitrarily.

### Example

For each customer, compute a running total, running order count and running maximum, and the cumulative share of
their revenue.

In [ ]:
so_far = in_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
orders.where("customer_id IN (1, 3, 4)").select(
    "customer_id", "order_id", "amount",
    F.sum("amount").over(so_far).alias("running_total"),
    F.count("*").over(so_far).alias("running_orders"),
    F.max("amount").over(so_far).alias("running_max"),
    F.round(F.sum("amount").over(so_far) / F.sum("amount").over(by_customer) * 100, 1).alias("cum_pct"),
).orderBy("customer_id", "order_id").show()

**What to notice:** Customer 1's running total grows 120.00, 204.50, 234.50 and its cumulative share reaches 100
percent on its last order. The running maximum never goes down: it jumps from 89.90 to 1,180.00 for customer 4
and from 149.00 to 310.00 for customer 3, then would stay there for any smaller later order.

## Moving windows: last N rows vs last N days

- `rowsBetween(-2, 0)`: this row and the two before it, a **3-row** moving window, whatever their dates.
- `rangeBetween(-2, 0)` over a day number: every row from the **last 3 calendar days**, however many rows.
- `rowsBetween(-1, 1)`: a centred window, smoothing each point with its neighbours.

To use `rangeBetween` with dates, order by a number: `F.unix_date("day")` (days since 1970-01-01) for dates, or
`F.col("ts").cast("long")` (seconds) for timestamps.

**Where it matters**
- **Smoothing dashboards** with a 7-day moving average.
- **Fraud and rate limits:** spend or attempts in the last 24 hours.

**Watch out for**
- A rows frame over data with missing days silently stretches across the gap; a range frame does not.
- The first rows of each partition average fewer values; filter them out if a full window is required.

### Example

Daily sales are missing 4 January. Compare a 3-row and a 3-day moving average, and a centred average.

In [ ]:
daily_sales = spark.createDataFrame(
    [(date(2026, 1, 1), 100), (date(2026, 1, 2), 200), (date(2026, 1, 3), 300),
     (date(2026, 1, 5), 500), (date(2026, 1, 6), 600)],  # no row for 4 January
    "day date, revenue int",
)
by_day_rows = Window.orderBy("day")
by_day_range = Window.orderBy(F.unix_date("day"))
daily_sales.select(
    "day", "revenue",
    F.avg("revenue").over(by_day_rows.rowsBetween(-2, 0)).alias("avg_3_rows"),
    F.avg("revenue").over(by_day_range.rangeBetween(-2, 0)).alias("avg_3_days"),
    F.count("*").over(by_day_range.rangeBetween(-2, 0)).alias("days_with_data"),
    F.avg("revenue").over(by_day_rows.rowsBetween(-1, 1)).alias("centred_3"),
).orderBy("day").show()

**What to notice:** On 5 January the 3-row average (333.33) reaches back to 2 and 3 January, across the missing
day. The 3-day range covers 3, 4 and 5 January, finds only two rows and averages 400. `days_with_data` makes that
visible, so a dashboard can flag incomplete windows. The centred average uses one row on each side.

## Fill gaps: forward fill and back fill

Sensors and slowly changing attributes often send a value only when it changes, leaving nulls in between.

- **Forward fill** (carry the last known value down): `F.last(col, ignorenulls=True)` over
  `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.
- **Back fill** (pull the next known value up): `F.first(col, ignorenulls=True)` over
  `rowsBetween(Window.currentRow, Window.unboundedFollowing)`.

**Where it matters**
- **IoT readings** and price feeds sent on change only.
- **Attributes** such as a customer's tier between updates.

**Watch out for**
- Forward fill cannot fill nulls before the first known value; decide whether to back fill those or leave them.
- Fill only when "unchanged since the last reading" is really what a null means.

### Example

Fill missing temperatures per sensor, forward and backward.

In [ ]:
readings = spark.createDataFrame(
    [("S1", 1, None), ("S1", 2, 20.5), ("S1", 3, None), ("S1", 4, None), ("S1", 5, 22.0),
     ("S2", 1, 18.0), ("S2", 2, None)],
    "sensor string, minute int, temp double",
)
by_sensor = Window.partitionBy("sensor").orderBy("minute")
readings.select(
    "sensor", "minute", "temp",
    F.last("temp", ignorenulls=True).over(by_sensor.rowsBetween(Window.unboundedPreceding, 0)).alias("forward_fill"),
    F.first("temp", ignorenulls=True).over(by_sensor.rowsBetween(0, Window.unboundedFollowing)).alias("back_fill"),
).orderBy("sensor", "minute").show()

**What to notice:** Minutes 3 and 4 of S1 carry 20.5 forward and pull 22.0 back, so the two methods disagree
inside a gap. S1's minute 1 has nothing before it, so only back fill can fill it; S2's minute 2 has nothing after
it, so only forward fill can. Combine them with `coalesce` if both edges must be filled.

## Sessionize events

A **session** is a burst of activity separated by inactivity, for example more than 30 minutes without an event.
The recipe combines three window steps:

1. `lag` gives the gap since the user's previous event;
2. `when` flags a new session when the gap is over the timeout or there is no previous event;
3. a running `sum` of the flags numbers the sessions.

Structured Streaming offers `session_window` for the same idea on streams
([lesson 11](pyspark-11-structured-streaming.ipynb)).

**Where it matters**
- **Web and app analytics:** sessions per user, session length, conversion per session.

**Watch out for**
- Partition by user, or sessions leak across users.
- Deduplicate retried events first, or a retry looks like activity.

### Example

Split each user's clicks into sessions with a 30-minute timeout, then summarize each session.

In [ ]:
clicks = spark.createDataFrame(
    [("u1", datetime(2026, 1, 6, 9, 0)), ("u1", datetime(2026, 1, 6, 9, 10)), ("u1", datetime(2026, 1, 6, 9, 25)),
     ("u1", datetime(2026, 1, 6, 11, 0)), ("u2", datetime(2026, 1, 6, 9, 0)), ("u2", datetime(2026, 1, 6, 9, 45))],
    "user_id string, ts timestamp",
)
user_time = Window.partitionBy("user_id").orderBy("ts")
sessions = (
    clicks
    .withColumn("gap_min", (F.col("ts").cast("long") - F.lag("ts").over(user_time).cast("long")) / 60)
    .withColumn("new_session", F.when(F.col("gap_min").isNull() | (F.col("gap_min") > 30), 1).otherwise(0))
    .withColumn("session", F.sum("new_session").over(user_time.rowsBetween(Window.unboundedPreceding, 0)))
)
sessions.orderBy("user_id", "ts").show()
sessions.groupBy("user_id", "session").agg(
    F.count("*").alias("clicks"),
    F.min("ts").alias("started"),
    ((F.max("ts").cast("long") - F.min("ts").cast("long")) / 60).alias("minutes"),
).orderBy("user_id", "session").show()

**What to notice:** u1's first three clicks are 10 and 15 minutes apart, so they form session 1; the 95-minute
gap before 11:00 starts session 2. u2's two clicks are 45 minutes apart, over the timeout, so each is its own
one-click session of 0 minutes.

## Find consecutive runs: gaps and islands

"Longest streak of consecutive active days" is a **gaps-and-islands** problem. Number each user's distinct days
with `row_number` and subtract that number from the date. Inside an unbroken run, the date and the row number both
go up by one each step, so the difference stays the same; grouping by it finds each run (an island).

**Where it matters**
- **Engagement:** login streaks, consecutive months with a purchase.
- **Operations:** periods of continuous downtime.

**Watch out for**
- Deduplicate to one row per user and day first, or a repeated day breaks the arithmetic.

### Example

Find each user's longest run of consecutive login days.

In [ ]:
logins = spark.createDataFrame(
    [("u1", "2026-01-01"), ("u1", "2026-01-02"), ("u1", "2026-01-02"), ("u1", "2026-01-03"),
     ("u1", "2026-01-05"), ("u1", "2026-01-06"), ("u2", "2026-01-10"), ("u3", "2026-01-01"), ("u3", "2026-01-03")],
    "user_id string, login_date string",
)
days = logins.select("user_id", F.to_date("login_date").alias("day")).distinct()
islands = days.withColumn("island", F.date_sub("day", F.row_number().over(Window.partitionBy("user_id").orderBy("day"))))
streaks = islands.groupBy("user_id", "island").agg(
    F.count("*").alias("days"), F.min("day").alias("start"), F.max("day").alias("end")
)
longest = Window.partitionBy("user_id").orderBy(F.desc("days"), "start")
streaks.withColumn("rn", F.row_number().over(longest)).where("rn = 1").drop("rn", "island").orderBy("user_id").show()

**What to notice:** u1's 1 to 3 January form one island and 5 to 6 January another, so the longest streak is 3
days; the duplicate login on the 2nd was removed by `distinct` first. u3's two separate days are two islands of
one day, and ordering by `start` makes the earlier one win the tie.

## Window functions in Spark SQL

The same windows in SQL use `OVER (PARTITION BY ... ORDER BY ... frame)`. A named `WINDOW` clause lets several
functions share one specification. Spark 4.0 has no `QUALIFY` clause, so filter on a window result in an outer
query or a CTE.

```sql
SELECT customer_id, order_id, amount,
       row_number() OVER w                                   AS rn,
       sum(amount)  OVER (w ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM orders
WINDOW w AS (PARTITION BY customer_id ORDER BY order_ts, order_id)
```

**Where it matters**
- **Analysts' queries** and dbt models that move into a Spark job.

**Watch out for**
- The same frame defaults apply in SQL: `ORDER BY` without a frame means `RANGE ... CURRENT ROW`.

### Example

Find each customer's largest order with a named window and an outer filter.

In [ ]:
spark.sql('''
    WITH ranked AS (
        SELECT customer_id, order_id, amount,
               row_number() OVER w AS rn,
               sum(amount) OVER (PARTITION BY customer_id) AS customer_total
        FROM {orders}
        WINDOW w AS (PARTITION BY customer_id ORDER BY amount DESC, order_id)
    )
    SELECT customer_id, order_id, amount, customer_total
    FROM ranked
    WHERE rn = 1
    ORDER BY customer_id
''', orders=orders).show()

**What to notice:** One row per customer, its largest order next to its total: the top-1 pattern in SQL. The CTE
computes `rn` and the outer query filters on it, which is how you write what other engines express with `QUALIFY`.

## How Spark runs a window, and how to keep it fast

For each window specification Spark **shuffles** rows by the `partitionBy` keys (an `Exchange`), **sorts** each
partition by the partition and order keys, then evaluates all functions that share that specification in one
`Window` operator. So:

- Functions over the **same** specification are computed together, in one shuffle and sort.
- Each **different** specification can add another sort, and another shuffle if its partition keys differ.
- A window **without `partitionBy`** moves every row to one partition (`Exchange SinglePartition`) and one task:
  fine for a small summary, a failure on a large table.
- One huge partition key (a "bot" user with millions of events) makes one task much slower than the rest.

**Where it matters**
- **Large fact tables,** where every extra shuffle costs minutes.

**Watch out for**
- Reuse a specification object for related columns rather than writing near-identical ones.
- Aggregate first when you only need one row per group; a window over billions of rows is more expensive.

### Example

Count the `Window` operators and shuffles for three window columns that share a specification, then for two
different specifications, and look for the single-partition shuffle.

In [ ]:
import contextlib
import io


def plan_summary(df):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain()
    lines = [line.strip() for line in buffer.getvalue().splitlines()]
    windows = sum(line.lstrip("+- ").startswith("Window ") for line in lines)
    exchanges = [line.split("Exchange ")[1].split(",")[0] for line in lines if "Exchange " in line]
    return f"Window operators: {windows}, shuffles: {exchanges}"


same_spec = orders.select(
    "*", F.row_number().over(in_order).alias("rn"), F.lag("amount").over(in_order).alias("prev"),
    F.sum("amount").over(in_order.rowsBetween(Window.unboundedPreceding, 0)).alias("running"),
)
two_specs = orders.select(
    "*", F.row_number().over(in_order).alias("rn"),
    F.rank().over(Window.partitionBy("status").orderBy("amount")).alias("status_rank"),
)
no_partition = orders.select("*", F.row_number().over(Window.orderBy("amount")).alias("overall_rank"))

print("same spec:   ", plan_summary(same_spec))
print("two specs:   ", plan_summary(two_specs))
print("no partition:", plan_summary(no_partition))

**What to notice:** The three columns that share `in_order` need one shuffle by `customer_id`; the frame on the
running total does not force another one. Adding a ranking partitioned by `status` adds a second shuffle and
`Window` operator. The window with no `partitionBy` shuffles to `SinglePartition`: every row goes to one task.

## Common mistakes

| symptom | cause | fix |
| --- | --- | --- |
| `last` returns the current row | default frame ends at the current row | `rowsBetween(unboundedPreceding, unboundedFollowing)` |
| a "group total" grows row by row | `orderBy` added to an aggregate's window | use a specification without `orderBy` |
| tied rows share a running total | default `RANGE` frame | `rowsBetween(...)` and a tie-breaker |
| results change between runs | `row_number` over non-unique ordering | add a unique column to `orderBy` |
| `lag` mixes customers | no `partitionBy` | partition by the entity |
| one task runs for hours | no `partitionBy`, or one huge key | partition the window; handle skewed keys separately |
| `AnalysisException` filtering on a window | window used inside `where` | compute it into a column, then filter |

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** When do you need a window function instead of `groupBy`?

**A.** When you must keep every input row and add a value computed from related rows, such as a rank, a group total on each row, or the previous row's value.

**Q2.** For revenues 300, 300, 200, what do `row_number`, `rank` and `dense_rank` return?

**A.** 1, 2, 3; 1, 1, 3; and 1, 1, 2.

**Q3.** You want "the two highest revenue values per region, with every rep who has them". Which function?

**A.** `dense_rank() <= 2`. `row_number` returns exactly two rows and `rank` can skip the second value after a tie.

**Q4.** What is the default frame for a window with `orderBy`, and which two surprises does it cause?

**A.** `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. `last` returns the current row, and rows that tie on the order key share a running total.

**Q5.** How do you forward fill a column with nulls?

**A.** `F.last(col, ignorenulls=True)` over the entity's window ordered by time, with `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

**Q6.** A 7-day moving average uses `rowsBetween(-6, 0)`. When is it wrong?

**A.** When days are missing: seven rows then span more than seven days. Use `rangeBetween(-6, 0)` over a day number such as `unix_date`.

**Q7.** How do you keep the latest version of each record reproducibly?

**A.** `row_number()` over `partitionBy(key).orderBy(source sequence desc, stable tie-breaker)`, then keep `rn = 1`.

**Q8.** Why can a window without `partitionBy` fail on a large table?

**A.** Spark moves every row into a single partition, processed by one task, which runs out of memory or takes too long.

## Key takeaways

- A window keeps every row: `partitionBy` says which rows belong together, `orderBy` their order, the frame which neighbours each row sees.
- `row_number`, `rank` and `dense_rank` differ only on ties; add a tie-breaker whenever the result must be stable.
- `lag` and `lead` compare a row with its neighbours; a change flag plus a running sum numbers runs and sessions.
- With `orderBy` and no frame, the frame stops at the current row: set `rowsBetween` explicitly for `last`, `nth_value` and running totals.
- Use rows frames for "the last N rows" and range frames over a number for "the last N days".
- Always partition large windows, and reuse one specification for related columns.

**Next:** Practise these patterns in [PySpark drills 7: aggregations and windows](pyspark-drills-07-aggregations-and-windows.ipynb), then continue with [PySpark 8: nested and semi-structured data](pyspark-08-nested-and-semi-structured-data.ipynb).